# P4-A4 — Reflection (agents that critique and improve their own work)

The final pattern: **self-correction**. Instead of accepting the model's first draft, you loop — *generate → critique → revise* — until the output is good enough (or you hit a cap). One model wears two hats: a **generator** and a **critic**.

This is the same loop shape you keep building (a graph that cycles until a stop condition), but here the cycle *improves quality* rather than calling tools. It's how you squeeze better output out of the same model.

Graph: `generate → reflect → (good? END : back to generate)`.

In [ ]:
# --- Setup (given) ---
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')
MAX_ITERS = 3

def chat(prompt):
    """One-shot LLM call, returns text."""
    return llm.invoke(prompt).content

# Shared state that flows through the loop
class ReflectState(TypedDict):
    task: str         # what we're trying to produce
    draft: str        # the current attempt
    critique: str     # the latest critique (empty on first pass)
    iterations: int   # how many drafts so far

print('ready — MAX_ITERS =', MAX_ITERS)

## Task 1 — The `generate` and `reflect` nodes
Write two node functions:
- **`generate(state)`** → produce a `draft`. On the first pass (no critique) just attempt the `task`; on later passes, *revise the previous draft using the critique*. Increment `iterations`. Print the draft.
- **`reflect(state)`** → critique the current `draft` against the `task`. If it's genuinely good, start the reply with `APPROVED`; otherwise give specific, actionable feedback. Store it in `critique`. Print it.
<br>Hint: each returns a dict updating only the keys it changes, e.g. `return {'draft': ..., 'iterations': state['iterations'] + 1}`.

In [ ]:
# TODO: def generate(state): ...   and   def reflect(state): ...


## Task 2 — Wire the reflection loop
Build the graph: `START → generate → reflect`, then a **conditional edge** from `reflect` that ends if the critique is `APPROVED` **or** `iterations >= MAX_ITERS`, otherwise loops back to `generate`.
<br>Write a `should_continue(state)` that returns `END` or `'generate'`, and compile to `app`.
<br>Hint: `g.add_conditional_edges('reflect', should_continue, {'generate': 'generate', END: END})`. The `MAX_ITERS` cap is your loop guard (same instinct as P2-A6 `max_steps` / P4-A3 `recursion_limit`).

In [ ]:
# TODO: should_continue(state) + build/compile the generate<->reflect graph as `app`


## Task 3 — Run it and watch the draft improve
Invoke `app` with a task that benefits from revision, e.g. *"Explain what an API is to a complete non-technical beginner, in 3 sentences, using a real-world analogy and no jargon."* Start state: `{'task': ..., 'draft': '', 'critique': '', 'iterations': 0}`.
<br>Watch the printed drafts + critiques across iterations, then print the final `draft`.
<br>Goal: see the output measurably improve from draft 1 → final.

In [ ]:
# TODO: run app on a revision-worthy task; observe drafts improving; print final draft


## Task 4 — Explain (in your own words)
1. Why can a model's *critique-then-revise* output beat its first attempt, even though it's the same model with no new information?
2. Reflection costs extra LLM calls per iteration. How would you decide *when* it's worth it, and how would you set the stopping condition well (think: the `APPROVED` signal, `MAX_ITERS`, and evals)?

> _your answer here_